# Python: kod, który bada kod
## AI Mastery — dr Marcin Albiniak

Osiem demonstracji do prezentacji „Język Python i jego środowiska”.

**Uruchom od góry:** Kernel → Restart Kernel and Run All Cells. Python 3.11 lub nowszy.

Moduły 01–05 i 07–08 używają biblioteki standardowej. Moduł 06 wymaga NumPy i scikit-learn, a rozszerzenie autograd opcjonalnie PyTorch. Brak tych bibliotek powoduje czytelny komunikat, a nie zatrzymanie całego notebooka.

Całość działa offline po przygotowaniu środowiska. W module 08 używamy **symulowanych propozycji**, nie prawdziwego wywołania LLM.

Przykłady są dydaktyczne. Fragment z eval w module 05 wykonuje wyłącznie własny, stały tekst. Nie jest sandboxem. Moduły 07–08 używają własnego interpretera małej gramatyki arytmetycznej.

In [1]:
import ast
import copy
import hashlib
import inspect
import json
import math
import operator
import platform
import random
import sys
from dataclasses import dataclass
from functools import wraps
from pathlib import Path

## 01. Obiekty, aliasy i kopie

Przewidź, czy zmiana b[0] będzie widoczna w a. Dlaczego deepcopy zachowuje poprzednią zawartość?

In [2]:
def demo01():
    a=[[1,2],[3]]
    b=a.copy()
    c=copy.deepcopy(a)
    b[0].append(9)
    assert a==b and a is not b and a[0] is b[0]
    assert c==[[1,2],[3]]
    return {'a':a,'b':b,'deepcopy':c,'python':platform.python_version(),'executable':sys.executable}

In [3]:
result01 = demo01()
print(json.dumps(result01, ensure_ascii=False, indent=2))

{
  "a": [
    [
      1,
      2,
      9
    ],
    [
      3
    ]
  ],
  "b": [
    [
      1,
      2,
      9
    ],
    [
      3
    ]
  ],
  "deepcopy": [
    [
      1,
      2
    ],
    [
      3
    ]
  ],
  "python": "3.12.14",
  "executable": "/opt/codex/runtimes/codex-primary-runtime/dependencies/python/bin/python3"
}


## 02. Dane i kontrola błędów

Które rekordy należy odrzucić? Dlaczego NaN wymaga osobnego sprawdzenia?

In [4]:
def demo02():
    records=[{'id':1,'amount':'12.5'},{'id':2,'amount':'7.5'},
             {'id':3,'amount':''},{'id':4,'amount':None},{'id':5,'amount':'NaN'}]
    accepted=[]
    rejected=[]
    for row in records:
        try:
            value=float(row['amount'])
            if not math.isfinite(value):
                raise ValueError('Wartość nie jest skończona')
            accepted.append(value)
        except (ValueError,TypeError) as exc:
            rejected.append({'id':row['id'],'reason':type(exc).__name__})
    assert sum(accepted)==20 and len(rejected)==3
    return {'accepted':accepted,'sum':sum(accepted),'rejected':rejected}

In [5]:
result02 = demo02()
print(json.dumps(result02, ensure_ascii=False, indent=2))

{
  "accepted": [
    12.5,
    7.5
  ],
  "sum": 20.0,
  "rejected": [
    {
      "id": 3,
      "reason": "ValueError"
    },
    {
      "id": 4,
      "reason": "TypeError"
    },
    {
      "id": 5,
      "reason": "ValueError"
    }
  ]
}


## 03. Funkcje, dekoratory i callable

Dodaj nową strategię do rejestru. Czy kod wywołujący musi się zmienić?

In [6]:
def demo03():
    log=[]
    def traced(fn):
        @wraps(fn)
        def wrapper(*args,**kwargs):
            log.append(fn.__name__)
            return fn(*args,**kwargs)
        return wrapper
    def make_threshold(limit):
        return lambda score:score>=limit
    @dataclass(frozen=True)
    class Threshold:
        limit:float
        def __call__(self,score):return score>=self.limit
    registry={'strict':traced(make_threshold(0.8)),
              'relaxed':make_threshold(0.4),'object':Threshold(0.7)}
    results={name:fn(0.75) for name,fn in registry.items()}
    assert results=={'strict':False,'relaxed':True,'object':True}
    return {'results':results,'calls':log}

In [7]:
result03 = demo03()
print(json.dumps(result03, ensure_ascii=False, indent=2))

{
  "results": {
    "strict": false,
    "relaxed": true,
    "object": true
  },
  "calls": [
    "<lambda>"
  ]
}


## 04. Introspekcja

Porównaj niepoprawne argumenty wywołania z niepoprawną wartością argumentu. Co sprawdza bind?

In [8]:
def demo04():
    def score(x:float,*,limit:float=0.5):return x>=limit
    sig=inspect.signature(score)
    good=dict(sig.bind(0.8,limit=0.7).arguments)
    try:sig.bind(limit=0.7)
    except TypeError as exc:binding_error=str(exc)
    # bind akceptuje zły typ: walidujemy go osobno.
    wrong=sig.bind('tekst',limit=0.7).arguments
    assert type(wrong['x']) is str
    @dataclass
    class Meter:
        reads:int=0
        @property
        def value(self):
            self.reads+=1
            return 42
    meter=Meter()
    members=inspect.getmembers_static(meter)
    before=meter.reads
    value=meter.value
    assert before==0 and meter.reads==1
    return {'signature':str(sig),'arguments':good,'binding_error':binding_error,
            'bind_does_not_validate_type':dict(wrong),'static_reads':before,'dynamic_reads':meter.reads}

In [9]:
result04 = demo04()
print(json.dumps(result04, ensure_ascii=False, indent=2))

{
  "signature": "(x: float, *, limit: float = 0.5)",
  "arguments": {
    "x": 0.8,
    "limit": 0.7
  },
  "binding_error": "missing a required argument: 'x'",
  "bind_does_not_validate_type": {
    "x": "tekst",
    "limit": 0.7
  },
  "static_reads": 0,
  "dynamic_reads": 1
}


## 05. AST i transformacja kodu

Zmień źródło na x + True. Dlaczego transformer pomija bool?

In [10]:
class DoubleInts(ast.NodeTransformer):
    def visit_Constant(self,node):
        if type(node.value) is int:
            return ast.copy_location(ast.Constant(value=node.value*2),node)
        return node

In [11]:
def demo05():
    source='x + 3'
    original=ast.parse(source,mode='eval')
    changed=ast.fix_missing_locations(DoubleInts().visit(copy.deepcopy(original)))
    # Tylko własny, literalny kod dydaktyczny. To NIE jest sandbox.
    value=eval(compile(changed,'<trusted-demo>','eval'),{'__builtins__':{}},{'x':4})
    bool_tree=DoubleInts().visit(ast.parse('True',mode='eval'))
    assert value==10 and bool_tree.body.value is True
    class Calls(ast.NodeVisitor):
        def __init__(self):self.names=[]
        def visit_Call(self,node):
            if isinstance(node.func,ast.Name):self.names.append(node.func.id)
            self.generic_visit(node)
    v=Calls();v.visit(ast.parse('print(abs(-3))'))
    assert v.names==['print','abs']
    return {'source':source,'AST':ast.dump(original,indent=2),'changed':ast.unparse(changed),
            'value_for_x4':value,'calls':v.names}

In [12]:
result05 = demo05()
print(json.dumps(result05, ensure_ascii=False, indent=2))

{
  "source": "x + 3",
  "AST": "Expression(\n  body=BinOp(\n    left=Name(id='x', ctx=Load()),\n    op=Add(),\n    right=Constant(value=3)))",
  "changed": "x + 6",
  "value_for_x4": 10,
  "calls": [
    "print",
    "abs"
  ]
}


In [13]:
import dis

def f(x):
    return x * x + 1

dis.dis(f)

  3           0 RESUME                   0

  4           2 LOAD_FAST                0 (x)
              4 LOAD_FAST                0 (x)
              6 BINARY_OP                5 (*)
             10 LOAD_CONST               1 (1)
             12 BINARY_OP                0 (+)
             16 RETURN_VALUE


## 06. Pipeline uczenia maszynowego

Dane są syntetyczne. Podział następuje przed dopasowaniem skalera. Porównaj z baseline i odczytaj macierz pomyłek.

In [14]:
def demo06():
    try:
        import numpy as np
        import sklearn
        from sklearn.datasets import make_classification
        from sklearn.model_selection import train_test_split
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler
        from sklearn.linear_model import LogisticRegression
        from sklearn.dummy import DummyClassifier
        from sklearn.metrics import accuracy_score, confusion_matrix
    except ImportError as exc:
        return {'status':'pominięto','reason':str(exc),'install':'python -m pip install numpy scikit-learn'}
    X,y=make_classification(n_samples=800,n_features=8,n_informative=5,n_redundant=1,
                            class_sep=1.2,random_state=42)
    X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
    model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000))
    model.fit(X_train,y_train)
    baseline=DummyClassifier(strategy='most_frequent').fit(X_train,y_train)
    pred=model.predict(X_test)
    assert pred.shape==y_test.shape and model[0].n_samples_seen_==len(X_train)
    return {'numpy':np.__version__,'sklearn':sklearn.__version__,'train_n':len(X_train),'test_n':len(X_test),
            'accuracy':accuracy_score(y_test,pred),'baseline_accuracy':baseline.score(X_test,y_test),
            'confusion_matrix':confusion_matrix(y_test,pred).tolist()}

In [15]:
result06 = demo06()
print(json.dumps(result06, ensure_ascii=False, indent=2))

{
  "numpy": "2.3.5",
  "sklearn": "1.8.0",
  "train_n": 600,
  "test_n": 200,
  "accuracy": 0.875,
  "baseline_accuracy": 0.5,
  "confusion_matrix": [
    [
      85,
      15
    ],
    [
      10,
      90
    ]
  ]
}


## 07. Ewolucja wyrażeń

Reprezentacja jest drzewem z operatorami +, - i *. Mutacja zmienia poddrzewo, crossover wymienia fragmenty. Test końcowy nie służy selekcji.

In [16]:
OPS={'+':operator.add,'-':operator.sub,'*':operator.mul}

AST_OPS={ast.Add:'+',ast.Sub:'-',ast.Mult:'*'}

MAX_NODES=63

MAX_DEPTH=6

MAX_ABS=1e8

def size(t):return 1 if len(t)<3 else 1+size(t[1])+size(t[2])

def depth(t):return 1 if len(t)<3 else 1+max(depth(t[1]),depth(t[2]))

def expression(t):
    if t[0]=='x':return 'x'
    if t[0]=='c':return str(t[1])
    return f'({expression(t[1])} {t[0]} {expression(t[2])})'

def calculate(t,x):
    if t[0]=='x':return float(x)
    if t[0]=='c':return float(t[1])
    value=OPS[t[0]](calculate(t[1],x),calculate(t[2],x))
    if not math.isfinite(value) or abs(value)>MAX_ABS:
        raise ValueError('Limit wyniku')
    return value

def parse_expression(source):
    if type(source) is not str or len(source)>240:
        raise ValueError('Limit długości źródła')
    tree=ast.parse(source,mode='eval')
    if sum(1 for _ in ast.walk(tree))>3*MAX_NODES:
        raise ValueError('Limit węzłów AST')
    def convert(node,level=1):
        if level>MAX_DEPTH:raise ValueError('Limit głębokości')
        if isinstance(node,ast.Name) and node.id=='x':return ('x',)
        if isinstance(node,ast.Constant) and type(node.value) in (int,float):
            if not math.isfinite(node.value) or abs(node.value)>10:raise ValueError('Limit stałej')
            return ('c',node.value)
        if isinstance(node,ast.BinOp) and type(node.op) in AST_OPS:
            return (AST_OPS[type(node.op)],convert(node.left,level+1),convert(node.right,level+1))
        if isinstance(node,ast.UnaryOp) and isinstance(node.op,ast.USub) and isinstance(node.operand,ast.Constant):
            v=node.operand.value
            if type(v) in (int,float) and math.isfinite(v) and abs(v)<=10:return ('c',-v)
        raise ValueError(f'Niedozwolony węzeł: {type(node).__name__}')
    candidate=convert(tree.body)
    if size(candidate)>MAX_NODES:raise ValueError('Limit rozmiaru')
    return candidate

def target(x):return x*x+2*x+1

TRAIN=[-2,-1.5,-1,-.5,0,.5,1,1.5,2]

TEST=[-3,-2.5,-1.75,-.25,.25,.75,1.75,2.5,3]

def mse(t,xs):
    try:return sum((calculate(t,x)-target(x))**2 for x in xs)/len(xs)
    except (ValueError,OverflowError):return 1e20

def fitness(t):return mse(t,TRAIN)+0.001*size(t)

def random_tree(rng,remaining=3):
    if remaining<=1 or rng.random()<.28:
        return ('x',) if rng.random()<.55 else ('c',rng.randint(-3,3))
    return (rng.choice(tuple(OPS)),random_tree(rng,remaining-1),random_tree(rng,remaining-1))

def paths(t,p=()):
    yield p
    if len(t)==3:
        yield from paths(t[1],p+(1,))
        yield from paths(t[2],p+(2,))

def subtree(t,p):
    for i in p:t=t[i]
    return t

def replace(t,p,new):
    if not p:return new
    result=list(t);result[p[0]]=replace(result[p[0]],p[1:],new)
    return tuple(result)

def bounded(child,parent):
    return child if size(child)<=MAX_NODES and depth(child)<=MAX_DEPTH else parent

def mutate(t,rng):
    p=rng.choice(list(paths(t)))
    child=replace(t,p,random_tree(rng,rng.randint(1,3)))
    return bounded(child,t)

def crossover(a,b,rng):
    pa=rng.choice(list(paths(a)));pb=rng.choice(list(paths(b)))
    return bounded(replace(a,pa,subtree(b,pb)),a)

In [17]:
def demo07(seed=42,population_size=120,generations=50):
    rng=random.Random(seed)
    population=[random_tree(rng,3) for _ in range(population_size)]
    history=[]
    evaluations=0
    for generation in range(generations+1):
        ranked=sorted(((fitness(t),t) for t in population),key=lambda z:z[0])
        evaluations+=len(population)
        best_cost,best=ranked[0]
        history.append({'generation':generation,'cost':best_cost,'train_mse':mse(best,TRAIN),
                        'nodes':size(best),'expression':expression(best)})
        if generation==generations:break
        def select():return min(rng.sample(ranked,5),key=lambda z:z[0])[1]
        next_population=[x[1] for x in ranked[:4]]
        while len(next_population)<population_size:
            parent=select()
            child=crossover(parent,select(),rng) if rng.random()<.65 else parent
            if rng.random()<.85:child=mutate(child,rng)
            next_population.append(child)
        population=next_population
    # TEST jest użyty dopiero po zakończeniu selekcji.
    final_test=mse(best,TEST)
    assert all(history[i+1]['cost']<=history[i]['cost']+1e-12 for i in range(len(history)-1))
    assert depth(best)<=MAX_DEPTH and size(best)<=MAX_NODES
    return {'seed':seed,'population':population_size,'generations':generations,'evaluations':evaluations,
            'best':expression(best),'train_mse':mse(best,TRAIN),'test_mse':final_test,'nodes':size(best),
            'history':history}

In [18]:
result07 = demo07()
print(json.dumps({k: v for k, v in result07.items() if k != "history"}, ensure_ascii=False, indent=2))
print("\nPierwsze pokolenia:")
for row in result07["history"][:8]:
    print(row)

{
  "seed": 42,
  "population": 120,
  "generations": 50,
  "evaluations": 6120,
  "best": "((x * x) + ((x + x) + 1))",
  "train_mse": 0.0,
  "test_mse": 0.0,
  "nodes": 9
}

Pierwsze pokolenia:
{'generation': 0, 'cost': 2.6716666666666664, 'train_mse': 2.6666666666666665, 'nodes': 5, 'expression': '((x * x) + x)'}
{'generation': 1, 'cost': 0.013000000000000001, 'train_mse': 0.0, 'nodes': 13, 'expression': '((x + ((3 + -3) + (x + 1))) + (x * x))'}
{'generation': 2, 'cost': 0.013000000000000001, 'train_mse': 0.0, 'nodes': 13, 'expression': '((x + ((3 + -3) + (x + 1))) + (x * x))'}
{'generation': 3, 'cost': 0.009000000000000001, 'train_mse': 0.0, 'nodes': 9, 'expression': '((x * x) + ((x + x) + 1))'}
{'generation': 4, 'cost': 0.009000000000000001, 'train_mse': 0.0, 'nodes': 9, 'expression': '((x * x) + ((x + x) + 1))'}
{'generation': 5, 'cost': 0.009000000000000001, 'train_mse': 0.0, 'nodes': 9, 'expression': '((x * x) + ((x + x) + 1))'}
{'generation': 6, 'cost': 0.009000000000000001, 't

**Eksperyment dodatkowy:** zmień seed lub wielkość populacji. Nie wybieraj najlepszej konfiguracji na podstawie wyniku testowego. Dokładne dopasowanie na kilku punktach nie gwarantuje poprawnej ekstrapolacji.

## 08. Kontrolowana synteza

Propozycje symulują wynik LLM. Nieznane węzły są odrzucane. Nie uruchamiamy dowolnego kodu Python.

In [19]:
def demo08():
    # Symulowane propozycje, bez rzeczywistego modelu i bez sieci.
    proposals=['x + 1','x * x + 2 * x + 1','abs(x)','x.__class__','True','1e100']
    report=[];valid=[]
    for source in proposals:
        identity=hashlib.sha256(source.encode()).hexdigest()[:12]
        try:
            t=parse_expression(source)
            cost=fitness(t)
            valid.append((cost,source,t))
            report.append({'id':identity,'source':source,'status':'oceniono','cost':cost})
        except (ValueError,SyntaxError,RecursionError) as exc:
            report.append({'id':identity,'source':source,'status':'odrzucono','reason':str(exc)})
    _,winner,t=min(valid,key=lambda z:z[0])
    test_mse=mse(t,TEST)
    assert winner=='x * x + 2 * x + 1' and test_mse==0
    assert sum(r['status']=='odrzucono' for r in report)==4
    return {'mode':'symulacja offline, brak wywołania LLM','report':report,'winner':winner,'test_mse':test_mse}

In [20]:
result08 = demo08()
print(json.dumps(result08, ensure_ascii=False, indent=2))

{
  "mode": "symulacja offline, brak wywołania LLM",
  "report": [
    {
      "id": "ab69983b9cb0",
      "source": "x + 1",
      "status": "oceniono",
      "cost": 6.586333333333333
    },
    {
      "id": "21869a744acc",
      "source": "x * x + 2 * x + 1",
      "status": "oceniono",
      "cost": 0.009000000000000001
    },
    {
      "id": "2fb978003dc1",
      "source": "abs(x)",
      "status": "odrzucono",
      "reason": "Niedozwolony węzeł: Call"
    },
    {
      "id": "f23ac3383e15",
      "source": "x.__class__",
      "status": "odrzucono",
      "reason": "Niedozwolony węzeł: Attribute"
    },
    {
      "id": "3cbc87c7681f",
      "source": "True",
      "status": "odrzucono",
      "reason": "Niedozwolony węzeł: Constant"
    },
    {
      "id": "f60559e05d36",
      "source": "1e100",
      "status": "odrzucono",
      "reason": "Limit stałej"
    }
  ],
  "winner": "x * x + 2 * x + 1",
  "test_mse": 0.0
}


## A4. Autograd (opcjonalnie)

Porównaj wynik z ręczną pochodną: d(x² + 2x + 1)/dx = 2x + 2. Dla x = 3 gradient wynosi 8. Ta sekcja wymaga PyTorch; przygotuj go przed zajęciami, jeżeli chcesz ją uruchamiać.

In [21]:
def extra_autograd():
    try:import torch
    except ImportError:return {'status':'pominięto, opcjonalny PyTorch'}
    x=torch.tensor(3.,requires_grad=True)
    y=x*x+2*x+1;y.backward()
    assert y.item()==16 and x.grad.item()==8
    return {'torch':torch.__version__,'y':y.item(),'gradient':x.grad.item()}

In [22]:
print(extra_autograd())

{'status': 'pominięto, opcjonalny PyTorch'}


## Wnioski

1. Nazwy i aliasy wyjaśniają mutację danych.
2. Funkcje i protokoły pozwalają wymieniać implementacje.
3. Introspekcja ujawnia interfejs, lecz nie dowodzi poprawności.
4. AST jest strukturą, którą można analizować i przekształcać.
5. Ewolucja programu wymaga reprezentacji, operatorów, oceny i budżetu.
6. Generator propozycji jest odrębny od wykonawcy i ewaluatora.

Dokumentacja: https://docs.python.org/3.14/library/ast.html, https://docs.python.org/3.14/library/inspect.html, https://scikit-learn.org/stable/common_pitfalls.html. Szczegółowe źródła i scenariusz wykładu znajdują się w notatkach prezentacji.